In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd

In [2]:
shap_eclipse = pd.read_csv("/kaggle/input/shap-weights/shap_weights_eclipse.csv")
shap_equinox = pd.read_csv("/kaggle/input/shap-weights/shap_weights_equinox.csv")
shap_lucene = pd.read_csv("/kaggle/input/shap-weights/shap_weights_lucene.csv")

In [4]:
print("Eclipse features:", shap_eclipse.shape)
print("Equinox features:", shap_equinox.shape)
print("Lucene features :", shap_lucene.shape)



Eclipse features: (5, 2)
Equinox features: (5, 2)
Lucene features : (5, 2)


In [5]:
set_eclipse = set(shap_eclipse["feature"])
set_equinox = set(shap_equinox["feature"])
set_lucene  = set(shap_lucene["feature"])

print("Only in Eclipse :", set_eclipse - set_equinox - set_lucene)
print("Only in Equinox :", set_equinox - set_eclipse - set_lucene)
print("Only in Lucene  :", set_lucene - set_eclipse - set_equinox)


Only in Eclipse : {'majorBugs'}
Only in Equinox : set()
Only in Lucene  : set()


In [6]:
common_features = set_eclipse & set_equinox & set_lucene

print("Number of common features:", len(common_features))
print("Common features:", sorted(common_features))


Number of common features: 4
Common features: ['numberOfBugsFoundUntil:', 'numberOfCriticalBugsFoundUntil:', 'numberOfMajorBugsFoundUntil:', 'numberOfNonTrivialBugsFoundUntil:']


In [7]:
shap_eclipse_f = shap_eclipse[shap_eclipse["feature"].isin(common_features)]
shap_equinox_f = shap_equinox[shap_equinox["feature"].isin(common_features)]
shap_lucene_f  = shap_lucene[shap_lucene["feature"].isin(common_features)]


In [8]:
shap_merged = (
    shap_eclipse_f
    .merge(shap_equinox_f, on="feature", suffixes=("_eclipse", "_equinox"))
    .merge(shap_lucene_f, on="feature")
    .rename(columns={"mean_abs_shap": "mean_abs_shap_lucene"})
)

shap_merged["avg_mean_abs_shap"] = (
    shap_merged["mean_abs_shap_eclipse"] +
    shap_merged["mean_abs_shap_equinox"] +
    shap_merged["mean_abs_shap_lucene"]
) / 3

shap_avg_weights = (
    shap_merged[["feature", "avg_mean_abs_shap"]]
    .sort_values(by="avg_mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_avg_weights
output_path = "/kaggle/working/shap_weights_avg_eel_common.csv"
shap_avg_weights.to_csv(output_path, index=False)

output_path


'/kaggle/working/shap_weights_avg_eel_common.csv'

Loading and pre-processing Mylyn dataset

In [11]:
import numpy as np

In [19]:
mylyn_raw = pd.read_csv("/kaggle/input/bug-prediction-dataset/bug-metrics-mylyn.csv", sep = ";")


mylyn_raw.columns = (
    mylyn_raw.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
)


mylyn_raw = mylyn_raw.select_dtypes(include=[np.number])


y_mylyn = (mylyn_raw["bugs"] > 0).astype(int)

# Features
X_mylyn = mylyn_raw.drop(columns=["bugs"]).fillna(0)


In [20]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
X_const = add_constant(X_mylyn)

vif_df = pd.DataFrame({
    "feature": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i)
            for i in range(X_const.shape[1])]
})

vif_df.sort_values("VIF", ascending=False)


,feature,VIF
1,numberOfBugsFoundUntil,30.883632
2,numberOfNonTrivialBugsFoundUntil,14.555507
5,numberOfHighPriorityBugsFoundUntil,13.427386
3,numberOfMajorBugsFoundUntil,3.744905
4,numberOfCriticalBugsFoundUntil,2.735035
7,majorBugs,1.575036
0,const,1.512039
9,highPriorityBugs,1.393011
8,criticalBugs,1.221844
6,nonTrivialBugs,1.204578


In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X_mylyn)

pca = PCA()
pca.fit(X_scaled)

explained_variance = np.cumsum(pca.explained_variance_ratio_)

explained_variance





array([0.49317573, 0.66776767, 0.7694404 , 0.85312764, 0.91784689,
       0.96582378, 0.98744625, 0.99765611, 1.        ])

In [27]:
skew_before = X_mylyn.skew().sort_values(ascending=False)
skew_before
X_mylyn_trans = np.log1p(X_mylyn)
skew_after = X_mylyn_trans.skew().sort_values(ascending=False)
skew_after

criticalBugs                          24.872975
majorBugs                             10.030767
highPriorityBugs                       7.231013
nonTrivialBugs                         3.966631
numberOfCriticalBugsFoundUntil         3.387979
numberOfMajorBugsFoundUntil            2.239970
numberOfBugsFoundUntil                 0.587511
numberOfNonTrivialBugsFoundUntil       0.514738
numberOfHighPriorityBugsFoundUntil     0.304478
dtype: float64

In [32]:
# Top 5 features fixed from SHAP-consensus phase
top5_features = [
    "numberOfBugsFoundUntil",
    "numberOfNonTrivialBugsFoundUntil",
    "numberOfMajorBugsFoundUntil",
    "numberOfCriticalBugsFoundUntil",
    "numberOfHighPriorityBugsFoundUntil"
]

# Safety check
missing = [f for f in top5_mylyn if f not in X_mylyn_trans.columns]
if missing:
    raise ValueError(f"Missing expected features in Mylyn dataset: {missing}")


In [33]:
X_mylyn_clean = X_mylyn_trans[top5_features].copy()


In [34]:
# Load averaged SHAP weights
shap_avg = pd.read_csv("/kaggle/working/shap_weights_avg_eel_common.csv")
shap_weight_map = dict(zip(shap_avg["feature"], shap_avg["avg_mean_abs_shap"]))
shap_weight_map

{'numberOfBugsFoundUntil:': 0.1309095874738779,
 'numberOfNonTrivialBugsFoundUntil:': 0.0580148606759513,
 'numberOfCriticalBugsFoundUntil:': 0.0094988290331483,
 'numberOfMajorBugsFoundUntil:': 0.0049329536789779}

In [35]:
X_mylyn_shap = X_mylyn_clean.copy()

for col in X_mylyn_shap.columns:
    if col in shap_weight_map:
        X_mylyn_shap[col] = X_mylyn_shap[col] * shap_weight_map[col]


In [36]:
print("Final Mylyn feature set:")
print(X_mylyn_shap.columns.tolist())

print("\nFeatures scaled by SHAP:")
print([f for f in top5_features if f in shap_weight_map])

print("\nFeatures left unscaled:")
print([f for f in top5_features if f not in shap_weight_map])


Final Mylyn feature set:
['numberOfBugsFoundUntil', 'numberOfNonTrivialBugsFoundUntil', 'numberOfMajorBugsFoundUntil', 'numberOfCriticalBugsFoundUntil', 'numberOfHighPriorityBugsFoundUntil']

Features scaled by SHAP:
[]

Features left unscaled:
['numberOfBugsFoundUntil', 'numberOfNonTrivialBugsFoundUntil', 'numberOfMajorBugsFoundUntil', 'numberOfCriticalBugsFoundUntil', 'numberOfHighPriorityBugsFoundUntil']


In [37]:
# Baseline-clean dataset
mylyn_clean_df = X_mylyn_clean.copy()
mylyn_clean_df["bugs"] = y_mylyn.values

# SHAP-weighted dataset
mylyn_shap_df = X_mylyn_shap.copy()
mylyn_shap_df["bugs"] = y_mylyn.values

# Save outputs
mylyn_clean_df.to_csv("/kaggle/working/mylyn_clean.csv", index=False)
mylyn_shap_df.to_csv("/kaggle/working/mylyn_shap_weighted.csv", index=False)

"/kaggle/working/mylyn_clean.csv", "/kaggle/working/mylyn_shap_weighted.csv"


('/kaggle/working/mylyn_clean.csv', '/kaggle/working/mylyn_shap_weighted.csv')

Loading and preprocessing Pde Data set

In [39]:
pde_raw = pd.read_csv("/kaggle/input/bug-prediction-dataset/bug-metrics-pde.csv", sep =";")

pde_raw.columns = (
    pde_raw.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
)

pde_raw = pde_raw.select_dtypes(include=[np.number])

y_pde = (pde_raw["bugs"] > 0).astype(int)

X_pde = pde_raw.drop(columns=["bugs"]).fillna(0)


In [40]:
X_pde_const = add_constant(X_pde)

vif_pde = pd.DataFrame({
    "feature": X_pde_const.columns,
    "VIF": [
        variance_inflation_factor(X_pde_const.values, i)
        for i in range(X_pde_const.shape[1])
    ]
}).sort_values("VIF", ascending=False)

vif_pde


/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,feature,VIF
2,numberOfNonTrivialBugsFoundUntil,25.152806
1,numberOfBugsFoundUntil,22.711977
3,numberOfMajorBugsFoundUntil,3.320448
4,numberOfCriticalBugsFoundUntil,2.640708
5,numberOfHighPriorityBugsFoundUntil,1.886239
6,nonTrivialBugs,1.413417
0,const,1.302187
7,majorBugs,1.229574
8,criticalBugs,1.167756
9,highPriorityBugs,NaN


In [41]:
X_pde_scaled = StandardScaler().fit_transform(X_pde)

pca = PCA()
pca.fit(X_pde_scaled)

explained_variance_pde = np.cumsum(pca.explained_variance_ratio_)
explained_variance_pde


array([0.4302028 , 0.61615264, 0.76627203, 0.85447939, 0.92119731,
       0.97133991, 0.9973482 , 1.        , 1.        ])

In [43]:
skew_pde_before = X_pde.skew().sort_values(ascending=False)
skew_pde_before


numberOfBugsFoundUntil                17.196563
criticalBugs                          15.716199
nonTrivialBugs                        13.073443
numberOfNonTrivialBugsFoundUntil      12.461247
numberOfHighPriorityBugsFoundUntil     8.627214
numberOfCriticalBugsFoundUntil         8.381499
majorBugs                              5.899542
numberOfMajorBugsFoundUntil            4.936547
highPriorityBugs                       0.000000
dtype: float64

In [44]:
X_pde_trans = np.log1p(X_pde)
skew_pde_after = X_pde_trans.skew().sort_values(ascending=False)
skew_pde_after


criticalBugs                          15.716199
nonTrivialBugs                        11.820658
numberOfHighPriorityBugsFoundUntil     5.627161
numberOfCriticalBugsFoundUntil         5.465150
majorBugs                              5.367681
numberOfMajorBugsFoundUntil            2.813302
numberOfNonTrivialBugsFoundUntil       0.643539
numberOfBugsFoundUntil                 0.520089
highPriorityBugs                       0.000000
dtype: float64

In [45]:
top5_features = [
    "numberOfBugsFoundUntil",
    "numberOfNonTrivialBugsFoundUntil",
    "numberOfMajorBugsFoundUntil",
    "numberOfCriticalBugsFoundUntil",
    "numberOfHighPriorityBugsFoundUntil"
]

# Safety check
missing = [f for f in top5_features if f not in X_pde_trans.columns]
if missing:
    raise ValueError(f"Missing expected features in PDE dataset: {missing}")


In [46]:
X_pde_clean = X_pde_trans[top5_features].copy()
shap_avg = pd.read_csv("/kaggle/working/shap_weights_avg_eel_common.csv")

shap_weight_map = dict(
    zip(shap_avg["feature"], shap_avg["avg_mean_abs_shap"])
)


In [47]:
X_pde_shap = X_pde_clean.copy()

for col in X_pde_shap.columns:
    if col in shap_weight_map:   # only 4 will match
        X_pde_shap[col] *= shap_weight_map[col]


In [48]:
print("Final PDE features:", X_pde_shap.columns.tolist())

print("Scaled by SHAP:",
      [f for f in top5_features if f in shap_weight_map])

print("Unscaled:",
      [f for f in top5_features if f not in shap_weight_map])


Final PDE features: ['numberOfBugsFoundUntil', 'numberOfNonTrivialBugsFoundUntil', 'numberOfMajorBugsFoundUntil', 'numberOfCriticalBugsFoundUntil', 'numberOfHighPriorityBugsFoundUntil']
Scaled by SHAP: []
Unscaled: ['numberOfBugsFoundUntil', 'numberOfNonTrivialBugsFoundUntil', 'numberOfMajorBugsFoundUntil', 'numberOfCriticalBugsFoundUntil', 'numberOfHighPriorityBugsFoundUntil']


In [49]:
pde_clean_df = X_pde_clean.copy()
pde_clean_df["bugs"] = y_pde.values

pde_shap_df = X_pde_shap.copy()
pde_shap_df["bugs"] = y_pde.values

pde_clean_df.to_csv("/kaggle/working/pde_clean.csv", index=False)
pde_shap_df.to_csv("/kaggle/working/pde_shap_weighted.csv", index=False)

"/kaggle/working/pde_clean.csv", "/kaggle/working/pde_shap_weighted.csv"


('/kaggle/working/pde_clean.csv', '/kaggle/working/pde_shap_weighted.csv')

In [51]:
!pip uninstall -y imbalanced-learn
!pip install imbalanced-learn==0.11.0


Found existing installation: imbalanced-learn 0.13.0
Uninstalling imbalanced-learn-0.13.0:
  Successfully uninstalled imbalanced-learn-0.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 6.2 MB/s eta 0:00:00:00:01


In [57]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import RandomOverSampler


In [58]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }


In [54]:
mylyn_shap = pd.read_csv("/kaggle/working/mylyn_shap_weighted.csv")
pde_shap   = pd.read_csv("/kaggle/working/pde_shap_weighted.csv")


In [59]:
def train_evaluate_dataset(df, dataset_name):
    X = df.drop(columns=["bugs"])
    y = df["bugs"]

    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )

    # Upsampling (TRAIN ONLY)
    ros = RandomOverSampler(random_state=42)
    X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "SVM (RBF)": SVC(probability=True),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            learning_rate=0.01,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.7,
            eval_metric="logloss",
            random_state=42
        )
    }

    results = []

    for name, model in models.items():
        model.fit(X_train_res, y_train_res)
        metrics = evaluate_model(model, X_test, y_test)
        metrics["Model"] = name
        metrics["Dataset"] = dataset_name
        results.append(metrics)

    return pd.DataFrame(results)


In [60]:
results_mylyn_shap = train_evaluate_dataset(
    mylyn_shap, dataset_name="Mylyn (SHAP-weighted)"
)

results_mylyn_shap


,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset
0,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,Mylyn (SHAP-weighted)
1,0.789700,0.305263,0.475410,0.371795,0.738980,SVM (RBF),Mylyn (SHAP-weighted)
2,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,Mylyn (SHAP-weighted)
3,0.738197,0.209524,0.360656,0.265060,0.532969,Decision Tree,Mylyn (SHAP-weighted)
4,0.733906,0.194175,0.327869,0.243902,0.591945,Random Forest,Mylyn (SHAP-weighted)
5,0.774678,0.284314,0.475410,0.355828,0.675875,Gradient Boosting,Mylyn (SHAP-weighted)
6,0.785408,0.298969,0.475410,0.367089,0.696074,XGBoost,Mylyn (SHAP-weighted)


In [63]:
results_pde_shap = train_evaluate_dataset(
    pde_shap, dataset_name="PDE (SHAP-weighted)"
)
results_pde_shap

,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset
0,0.690667,0.271429,0.730769,0.395833,0.801411,Logistic Regression,PDE (SHAP-weighted)
1,0.712000,0.287879,0.730769,0.413043,0.788075,SVM (RBF),PDE (SHAP-weighted)
2,0.842667,0.428571,0.403846,0.415842,0.717641,KNN,PDE (SHAP-weighted)
3,0.674667,0.217742,0.519231,0.306818,0.563825,Decision Tree,PDE (SHAP-weighted)
4,0.674667,0.234848,0.596154,0.336957,0.667123,Random Forest,PDE (SHAP-weighted)
5,0.680000,0.242424,0.615385,0.347826,0.704394,Gradient Boosting,PDE (SHAP-weighted)
6,0.656000,0.234483,0.653846,0.345178,0.759734,XGBoost,PDE (SHAP-weighted)


Baesline Models result

In [65]:
mylyn_baseline = pd.read_csv("/kaggle/working/mylyn_clean.csv")
results_mylyn_baseline = train_evaluate_dataset(
    mylyn_baseline, dataset_name="MYLYN (baseline)"
)
results_mylyn_baseline

,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset
0,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,MYLYN (baseline)
1,0.789700,0.305263,0.475410,0.371795,0.739061,SVM (RBF),MYLYN (baseline)
2,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,MYLYN (baseline)
3,0.738197,0.209524,0.360656,0.265060,0.532969,Decision Tree,MYLYN (baseline)
4,0.733906,0.194175,0.327869,0.243902,0.591945,Random Forest,MYLYN (baseline)
5,0.774678,0.284314,0.475410,0.355828,0.675875,Gradient Boosting,MYLYN (baseline)
6,0.785408,0.298969,0.475410,0.367089,0.696074,XGBoost,MYLYN (baseline)


In [66]:
pde_baseline = pd.read_csv("/kaggle/working/pde_clean.csv")
results_pde_baseline = train_evaluate_dataset(
    pde_baseline, dataset_name="PDE (baseline)"
)
results_pde_baseline

,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset
0,0.690667,0.271429,0.730769,0.395833,0.801411,Logistic Regression,PDE (baseline)
1,0.712000,0.287879,0.730769,0.413043,0.788075,SVM (RBF),PDE (baseline)
2,0.842667,0.428571,0.403846,0.415842,0.717641,KNN,PDE (baseline)
3,0.674667,0.217742,0.519231,0.306818,0.563825,Decision Tree,PDE (baseline)
4,0.674667,0.234848,0.596154,0.336957,0.667123,Random Forest,PDE (baseline)
5,0.680000,0.242424,0.615385,0.347826,0.704394,Gradient Boosting,PDE (baseline)
6,0.656000,0.234483,0.653846,0.345178,0.759734,XGBoost,PDE (baseline)


In [67]:
all_results = pd.concat([
    results_mylyn_baseline,
    results_mylyn_shap,
    results_pde_baseline,
    results_pde_shap
], ignore_index=True)

all_results


,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset
0,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,MYLYN (baseline)
1,0.789700,0.305263,0.475410,0.371795,0.739061,SVM (RBF),MYLYN (baseline)
2,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,MYLYN (baseline)
3,0.738197,0.209524,0.360656,0.265060,0.532969,Decision Tree,MYLYN (baseline)
4,0.733906,0.194175,0.327869,0.243902,0.591945,Random Forest,MYLYN (baseline)
5,0.774678,0.284314,0.475410,0.355828,0.675875,Gradient Boosting,MYLYN (baseline)
6,0.785408,0.298969,0.475410,0.367089,0.696074,XGBoost,MYLYN (baseline)
7,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,Mylyn (SHAP-weighted)
8,0.789700,0.305263,0.475410,0.371795,0.738980,SVM (RBF),Mylyn (SHAP-weighted)
9,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,Mylyn (SHAP-weighted)


In [68]:
all_results["Data_Type"] = all_results["Dataset"].apply(
    lambda x: "SHAP-weighted" if "SHAP" in x else "Baseline"
)

all_results


,Accuracy,Precision,Recall,F1-score,ROC-AUC,Model,Dataset,Data_Type
0,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,MYLYN (baseline),Baseline
1,0.789700,0.305263,0.475410,0.371795,0.739061,SVM (RBF),MYLYN (baseline),Baseline
2,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,MYLYN (baseline),Baseline
3,0.738197,0.209524,0.360656,0.265060,0.532969,Decision Tree,MYLYN (baseline),Baseline
4,0.733906,0.194175,0.327869,0.243902,0.591945,Random Forest,MYLYN (baseline),Baseline
5,0.774678,0.284314,0.475410,0.355828,0.675875,Gradient Boosting,MYLYN (baseline),Baseline
6,0.785408,0.298969,0.475410,0.367089,0.696074,XGBoost,MYLYN (baseline),Baseline
7,0.770386,0.260417,0.409836,0.318471,0.695608,Logistic Regression,Mylyn (SHAP-weighted),SHAP-weighted
8,0.789700,0.305263,0.475410,0.371795,0.738980,SVM (RBF),Mylyn (SHAP-weighted),SHAP-weighted
9,0.789700,0.271605,0.360656,0.309859,0.626068,KNN,Mylyn (SHAP-weighted),SHAP-weighted


In [69]:
mylyn_compare = all_results[
    all_results["Dataset"].str.contains("Mylyn")
].pivot_table(
    index="Model",
    columns="Data_Type",
    values=["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
)

mylyn_compare


,Accuracy,F1-score,Precision,ROC-AUC,Recall
Data_Type,SHAP-weighted,SHAP-weighted,SHAP-weighted,SHAP-weighted,SHAP-weighted
Model,,,,,
Decision Tree,0.738197,0.265060,0.209524,0.532969,0.360656
Gradient Boosting,0.774678,0.355828,0.284314,0.675875,0.475410
KNN,0.789700,0.309859,0.271605,0.626068,0.360656
Logistic Regression,0.770386,0.318471,0.260417,0.695608,0.409836
Random Forest,0.733906,0.243902,0.194175,0.591945,0.327869
SVM (RBF),0.789700,0.371795,0.305263,0.738980,0.475410
XGBoost,0.785408,0.367089,0.298969,0.696074,0.475410
